In [ ]:
from pathlib import Path
from urllib.request import urlopen
import shutil

def download_pretrained(option):
    """
    You can use this to download GloVe (or similar) model using wget.
    It's fine to download it through other means, but don't unzip the file (the rest of the notebook can
    handle the file in its zipped format).

    This function returns the path which you can use for the loader class that follows.

    The function will skip the download if it detects the file in disk. If the file in disk is corrupted, you will have to delete the file first.
    """
    if option == 'glove':
        destination = "glove.2024.wikigiga.50d.zip"
        dims = 50
        skip_first = False
    elif option == 'fasttext':
        destination = "wiki-news-300d-1M.vec.zip"
        dims = 300
        skip_first = True
    else:
        raise ValueError(f"Unkonwn option: {option}")

    if not Path(destination).is_file():
        if option == 'glove':
            !wget https://nlp.stanford.edu/data/wordvecs/glove.2024.wikigiga.50d.zip
        elif option == 'fasttext':
            !wget https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip


    return destination, dims, skip_first

In [13]:
import course_code.part2 as part2

# depends on internet connection (but say 5 minutes or less)
embtable_version_ft, embtable_dims_ft, embtable_metadata_ft = download_pretrained('fasttext') # in case stanford.edu is down, use fasttext (but glove is preferred)

trained_emb_model_ft = part2.clslib.PretrainedEmbeddings(embtable_version_ft, embtable_dims_ft, skip_first_line=embtable_metadata_ft)

In [6]:
import nltk
nltk.download("wordnet")

[nltk_data] Downloading package wordnet to /home/owen/nltk_data...


True

In [ ]:
import random
import numpy as np
from nltk.corpus import wordnet as wn   # pip install nltk; nltk.download("wordnet")
from sklearn.metrics import roc_auc_score
from collections import defaultdict

random.seed(0)

def cos(model, a, b):
    va, vb = model.embed_word(a), model.embed_word(b)
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb) + 1e-9))

def testing_antonym_synonym(model):
    vocab = model.vocab
    def clean(lemma):
        """Return lowercase single-word lemma name if it's in the vocab, else None."""
        name = lemma.name().lower()
        return name if "_" not in name and name in vocab else None

    syns = defaultdict(set)
    ants = defaultdict(set)

    for synset in wn.all_synsets():
        # Synonyms: all valid lemmas in the same synset
        words = {w for w in map(clean, synset.lemmas()) if w}
        for w in words:
            syns[w] |= words - {w}

        # Antonyms: lemma-level relation, so check each lemma and its antonym lemmas
        for lemma in synset.lemmas():
            w = clean(lemma)
            if w is None:
                continue
            for ant_lemma in lemma.antonyms():
                a = clean(ant_lemma)
                if a is None or a == w:
                    continue
                ants[w].add(a)
                ants[a].add(w)   # keep it symmetric

    # Drop empty entries and convert to plain dicts
    syns = {w: s for w, s in syns.items() if s}
    ants = {w: s for w, s in ants.items() if s}

    # --- Test 1: synonyms vs. random pairs ---
    synonyms = [(w, s) for w, ss in syns.items() for s in ss if w < s]
    words = model.vocab.words
    random_pairs = [(random.choice(words), random.choice(words)) for _ in range(len(synonyms))]
    scores_synonyms = [cos(model, a, b) for a, b in synonyms]
    scores_random_pairs = [cos(model, a, b) for a, b in random_pairs]
    scores = scores_synonyms + scores_random_pairs
    labels = [1] * len(synonyms) + [0] * len(random_pairs)
    print("--- Synonyms vs Random Pairs ---")
    print("ROC-AUC:", roc_auc_score(labels, scores))
    print("mean cos synonyms:", np.mean(scores[:len(synonyms)]),
        "| random:", np.mean(scores[len(synonyms):]))

    # --- Test 2: Antonyms vs. random pairs ---
    antonyms = [(w, s) for w, ss in ants.items() for s in ss if w < s]
    words = model.vocab.words
    scores_antonyms = [cos(model, a, b) for a, b in antonyms]
    scores = scores_antonyms + scores_random_pairs
    labels = [1] * len(antonyms) + [0] * len(random_pairs)
    print("--- Antonyms vs Random Pairs ---")
    print("ROC-AUC:", roc_auc_score(labels, scores))
    print("mean cos synonyms:", np.mean(scores[:len(antonyms)]),
        "| random:", np.mean(scores[len(antonyms):]))

    # --- Test 3: Antonyms vs. Synonyms ---
    print(len(synonyms), len(antonyms))
    if len(synonyms) > len(antonyms):
        scores_synonyms = scores_synonyms[:len(antonyms)]
    elif len(antonyms) > len(synonyms):
        scores_antonyms = scores_antonyms[:len(synonyms)]

    scores = scores_synonyms + scores_antonyms
    labels = [1] * len(scores_synonyms) + [0] * len(scores_antonyms)
    print("--- Antonyms vs Synonyms ---")
    print("ROC-AUC:", roc_auc_score(labels, scores))
    print("mean cos synonyms:", np.mean(scores[:len(antonyms)]),
        "| mean cos antonyms:", np.mean(scores[len(antonyms):]))

testing_antonym_synonym(trained_emb_model_ft)


--- Synonyms vs Random Pairs
ROC-AUC: 0.8336239576454529
mean cos synonyms: 0.47300897933656877 | random: 0.2729284106811012
--- Antonyms vs Random Pairs ---
ROC-AUC: 0.9648272293065048
mean cos synonyms: 0.5990443720408078 | random: 0.2729284106811012
59463 2786
---------Antonyms vs Synonyms--------------
ROC-AUC: 0.2881195924242276
mean cos synonyms: 0.48787078447238796 | mean cos antonyms: 0.5990443720408078


In [21]:
import zipfile
with zipfile.ZipFile("glove.6B.zip") as z:
    z.extract("glove.6B.300d.txt")

In [23]:
# Doing the same for glove
trained_emb_model_gl = part2.clslib.PretrainedEmbeddings("glove.6B.300d.txt", num_dims=300, skip_first_line=False)


In [48]:
testing_antonym_synonym(trained_emb_model_gl)

--- Synonyms vs Random Pairs
ROC-AUC: 0.7156244067646171
mean cos synonyms: 0.23610145854144154 | random: 0.10273522083053022
--- Antonyms vs Random Pairs ---
ROC-AUC: 0.899407355742185
mean cos synonyms: 0.368584812404968 | random: 0.10273522083053022
50027 2352
---------Antonyms vs Synonyms--------------
ROC-AUC: 0.3141639840459993
mean cos synonyms: 0.24004261800597124 | mean cos antonyms: 0.368584812404968


In [51]:
trained_emb_model_nb = part2.clslib.PretrainedEmbeddings("numberbatch-en-19.08.txt.gz", num_dims=300, skip_first_line=False)

In [52]:
testing_antonym_synonym(trained_emb_model_nb)

--- Synonyms vs Random Pairs
ROC-AUC: 0.9786663599739915
mean cos synonyms: 0.5419309079735805 | random: 0.009687886042781587
--- Antonyms vs Random Pairs ---
ROC-AUC: 0.9943588828998421
mean cos synonyms: 0.5548519285207276 | random: 0.009687886042781587
66283 3217
---------Antonyms vs Synonyms--------------
ROC-AUC: 0.4190623928347703
mean cos synonyms: 0.4862388209024381 | mean cos antonyms: 0.5548519285207276


In [ ]:
from course_code.part2.clslib import EmbedAndPoolEncoder
EmbedAndPoolEncoder(trained_emb_model_nb.vectors)